In [10]:
"""
Train an SVM on the source dataset and evaluate
on (1) source test split, (2) target dataset
without coral, and (3) target dataset with coral.
"""
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

In [11]:
# Load datasets and pretrained artifacts
source_train_data_path = os.path.join('data', 'processed', 'source', 'train.csv')
source_test_data_path = os.path.join('data', 'processed', 'source', 'test.csv')
target_train_data_path = os.path.join('data', 'processed', 'target', 'train.csv')
target_test_data_path = os.path.join('data', 'processed', 'target', 'test.csv')

label_encoder_path = os.path.join('models', 'label_encoder.joblib')
coral_source_stats_path = os.path.join('models', 'coral_source_stats.joblib')
coral_target_stats_path = os.path.join('models', 'coral_target_stats.joblib')

source_train_df = pd.read_csv(source_train_data_path)
source_test_df = pd.read_csv(source_test_data_path)
target_train_df = pd.read_csv(target_train_data_path)
target_test_df = pd.read_csv(target_test_data_path)

label_encoder = joblib.load(label_encoder_path)
coral_source_stats = joblib.load(coral_source_stats_path)
coral_target_stats = joblib.load(coral_target_stats_path)

print(f"Source train shape: {source_train_df.shape}")
print(f"Source test shape: {source_test_df.shape}")
print(f"Target train shape: {target_train_df.shape}")
print(f"Target test shape: {target_test_df.shape}")
print("Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats")

Source train shape: (2262141, 69)
Source test shape: (565536, 69)
Target train shape: (4280466, 69)
Target test shape: (1070117, 69)
Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats


In [15]:
# Sanity checks
# Verify source and target datasets have equivalent feature and label space.

shared_feature_path = os.path.join('data', 'processed', 'shared_feature_space.json')
shared_label_path = os.path.join('data', 'processed', 'shared_label_space.json')

import json
with open(shared_feature_path, 'r') as f:
    shared_features = json.load(f)
with open(shared_label_path, 'r') as f:
    shared_labels = json.load(f)

assert set(shared_features) <= set(source_train_df.columns), "Source train missing shared features!"
assert set(shared_features) <= set(source_test_df.columns), "Source test missing shared features!"
assert set(shared_features) <= set(target_train_df.columns), "Target train missing shared features!"
assert set(shared_features) <= set(target_test_df.columns), "Target test missing shared features!"

# Normalize each split's labels into class-name space before set comparison.
def to_label_name_set(label_series, fitted_label_encoder):
    labels = label_series.dropna()
    if pd.api.types.is_numeric_dtype(labels):
        return set(fitted_label_encoder.inverse_transform(labels.astype(int).to_numpy()))
    return set(labels.astype(str).to_numpy())

# use helper functio and pass label encoder
expected_label_set = set(map(str, shared_labels))
source_train_label_set = to_label_name_set(source_train_df['Label'], label_encoder)
source_test_label_set = to_label_name_set(source_test_df['Label'], label_encoder)
target_train_label_set = to_label_name_set(target_train_df['Label'], label_encoder)
target_test_label_set = to_label_name_set(target_test_df['Label'], label_encoder)

source_combined_label_set = source_train_label_set | source_test_label_set
target_combined_label_set = target_train_label_set | target_test_label_set

assert source_combined_label_set == expected_label_set, (
    "Combined source label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - source_combined_label_set)}, "
    f"Extra={sorted(source_combined_label_set - expected_label_set)}"
 )
assert target_combined_label_set == expected_label_set, (
    "Combined target label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - target_combined_label_set)}, "
    f"Extra={sorted(target_combined_label_set - expected_label_set)}"
 )
assert source_combined_label_set == target_combined_label_set, (
    "Combined source/target label spaces do not match!"
 )

print("Sanity checks passed: Shared feature space verified for source train/test and target train/test.")
print("Sanity checks passed: Combined source and target label spaces match shared labels.")

Sanity checks passed: Shared feature space verified for source train/test and target train/test.
Sanity checks passed: Combined source and target label spaces match shared labels.


In [9]:
### Fit SVM on source dataset (train split only) ###

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label']

# Use loaded label encoder (data is already scaled)
y_source_train_enc = label_encoder.transform(y_source_train)

# Train SVM
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_source_train, y_source_train_enc)

print("SVM trained on source train.csv dataset using loaded encoder.")

NameError: name 'source_train_df' is not defined

In [ ]:
### Evaluate SVM ###
# (1) Evaluate SVM on source test split

source_test_path = os.path.join('data', 'processed', 'source', 'test.csv')
source_test_df = pd.read_csv(source_test_path)

X_source_test = source_test_df[shared_features]
y_source_test = source_test_df['Label']

y_source_test_enc = label_encoder.transform(y_source_test)

y_source_test_pred = svm.predict(X_source_test)

print('Source Test Accuracy:', accuracy_score(y_source_test_enc, y_source_test_pred))
print('\nSource Test Classification Report:')
print(classification_report(y_source_test_enc, y_source_test_pred, target_names=label_encoder.classes_))

In [ ]:
### Evaluate SVM ###
# (2) Evaluate SVM on target test split WITHOUT CORAL domain adaptation (CIC_ToN_IoT)

X_target = target_test_df[shared_features]
y_target = target_test_df['Label']

y_target_enc = label_encoder.transform(y_target)

y_target_pred = svm.predict(X_target)

print('Target Test Accuracy (No CORAL):', accuracy_score(y_target_enc, y_target_pred))
print('\nTarget Test Classification Report (No CORAL):')
print(classification_report(y_target_enc, y_target_pred, target_names=label_encoder.classes_))

In [ ]:
### Evaluate SVM ###
# (3) Evaluate SVM on target test split WITH CORAL domain adaptation (CIC_ToN_IoT)

from scipy.linalg import fractional_matrix_power

# Validate and load source/target CORAL statistics
source_feature_order = coral_source_stats.get('feature_order')
target_feature_order = coral_target_stats.get('feature_order')

if source_feature_order is None or target_feature_order is None:
    raise KeyError("CORAL stats must include 'feature_order' metadata.")
if list(source_feature_order) != list(target_feature_order):
    raise ValueError("Source and target CORAL stats feature_order do not match.")
if list(shared_features) != list(source_feature_order):
    raise ValueError(
        "Runtime shared feature order does not match CORAL stats feature_order. "
        "Regenerate stats or align feature ordering before evaluation."
    )

source_mean = np.asarray(coral_source_stats['mean'])
source_cov = np.asarray(coral_source_stats['covariance'])

target_mean = np.asarray(coral_target_stats['mean'])
target_cov = np.asarray(coral_target_stats['covariance'])

n_features = len(source_feature_order)
if source_mean.shape[0] != n_features or target_mean.shape[0] != n_features:
    raise ValueError("CORAL mean vector length does not match feature_order length.")
if source_cov.shape != (n_features, n_features) or target_cov.shape != (n_features, n_features):
    raise ValueError("CORAL covariance shape does not match feature_order length.")

# Convert target features to numpy using the validated feature order
X_target_np = X_target[source_feature_order].to_numpy(dtype=np.float32)

# Center target data using target mean
X_target_centered = X_target_np - target_mean

# Compute CORAL transform:
# A = Ct^(-1/2) * Cs^(1/2)
target_cov_inv_sqrt = fractional_matrix_power(target_cov, -0.5)
source_cov_sqrt = fractional_matrix_power(source_cov, 0.5)

A_coral = target_cov_inv_sqrt @ source_cov_sqrt

# Apply CORAL transform
X_target_coral = X_target_centered @ A_coral

# Optional mean shift into source space
X_target_coral = X_target_coral + source_mean

# Predict using adapted target data
y_target_coral_pred = svm.predict(X_target_coral)

print('Target Test Accuracy (WITH CORAL):', accuracy_score(y_target_enc, y_target_coral_pred))
print('\nTarget Test Classification Report (WITH CORAL):')
print(
    classification_report(
        y_target_enc,
        y_target_coral_pred,
        target_names=label_encoder.classes_
    )
)